# PROJECT REQUIREMENTS & DEPENDENCIES
This project requires **Python 3.11** and specialized libraries optimized for **GPU inference (NVIDIA RTX 4060)**.

### Core Dependencies:
* **Deep Learning & Computer Vision:** `torch`, `torchvision` (CUDA 12.4 support), `ultralytics`
* **Data Processing:** `numpy`, `pandas`
* **Deployment & Backend:** `Flask`, `flask-cors`

> 💡 **Note:** If you are running this project for the first time on the GPU workstation, you can use the automatic installer cell below by uncommenting the last line.

In [ ]:
import sys  # Import system-specific parameters and functions to locate the active Python executable
import subprocess  # Import subprocess module to execute system-level terminal commands directly from the code

def install_requirements():  # Define a dedicated automated function to verify and install all project dependencies
    print("⏳ Starting installation, this may take a few minutes...")  # Output an installation initiation alert status text to the user
    
    # Download and install PyTorch and TorchVision with explicit CUDA 12.4 binary support optimized for RTX 4060 hardware acceleration
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu124"])
    
    # Define an absolute list array containing all essential computer vision and backend server deployment runtime packages
    requirements = ["opencv-python", "numpy", "pandas", "ultralytics", "facenet-pytorch", "pillow", "Flask", "flask-cors"]
    
    for package in requirements:  # Loop through every individual library package package name specified inside the requirements collection
        # Invoke the system package installer wrapper command to pull and configure each component inside the active execution environment
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        
    print("✅ Installation completed successfully! You can now run the cells below.")  # Render a final processing success completion notification text

# install_requirements()  # Uncomment this execution line only if you need to trigger a complete environment rebuilding setup

# CELL 1 — ENVIRONMENT SETUP & GPU FORCE
# Run this cell FIRST and alone

In [ ]:
import os 
import json  
import time  
import cv2  
import torch 
import numpy as np  
from threading import Thread 
from ultralytics import YOLO  
# Prevent initialization crashes related to duplicate OpenMP libraries inside the notebook runtime
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Evaluate hardware ecosystem to check if a compatible Nvidia CUDA GPU is available on the local machine
if torch.cuda.is_available():
    torch.cuda.set_device(0)  # Bind execution explicitly to the first available graphics card index
    torch.cuda.empty_cache()  # Clear unused allocated memory from the GPU to optimize dynamic VRAM spaces
    DEVICE_IDX = 0  # Store the active GPU index integer for references inside the model engine
    DEVICE_NAME = torch.cuda.get_device_name(0)  # Retrieve the human-readable hardware descriptor name of the GPU
else:
    DEVICE_IDX = "cpu"  # Fallback to standard CPU processing index if no Nvidia graphics card is detected
    DEVICE_NAME = "CPU"  # Set hardware descriptor name to CPU for pipeline logging clarity

CONFIG_PATH = "config.json"  # Define the standard local text file path for reading user boundary configurations
TELEMETRY_PATH = "telemetry.json"  # Define the output file path where counting analytics statistics are saved
UDP_URL = "udp://0.0.0.0:9999?listen&timeout=2000000"  # Set up connection socket string with a 2-second timeout parameter

print(f"[INFO] Initial hardware acceleration environment configured successfully on device: {DEVICE_NAME}")


🚀 GPU LOCKED → [0] NVIDIA GeForce RTX 4060



# CELL 2 — BACKGROUND STREAM READER & HELPERS

In [ ]:
class RTSPVideoStream:  # Define a dedicated class to handle non-blocking real-time video streaming over network protocols
    def __init__(self, src):  # Initialize the stream reader class constructor with the source UDP endpoint URL
        self.stream = cv2.VideoCapture(src, cv2.CAP_FFMPEG)  # Open the video stream using FFMPEG decoding backend wrapper
        self.stream.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Force internal hardware buffer stack to minimum size of 1 frame
        self.stopped = False  # Set boolean control flag to manage the background worker thread execution life cycle
        self.ret = False  # Initialize the frame retrieval success status indicator flag to structural False baseline
        self.frame = None  # Initialize frame variable storage container to empty state holds no image array matrix yet

    def start(self):  # Define start function to kick off parallel background thread execution routines
        t = Thread(target=self.update, args=())  # Instantiate a separate parallel worker thread targeting frame update loop
        t.daemon = True  # Mark thread as background daemon so it exits automatically when main notebook kernel terminates
        t.start()  # Launch the thread execution path to begin reading network packets immediately
        return self  # Return instance object reference to allow chained command method execution patterns down the line

    def update(self):  # Define background execution loop that constantly drains data layers from the network socket connection
        while True:  # Enter infinite loop structure to constantly listen for incoming video transmission packets
            if self.stopped:  # Evaluate if external program instruction sequence has requested thread termination routines
                break  # Break out of execution loop immediately to safely close worker process down without leaks
            self.ret, self.frame = self.stream.read()  # Grab and fully decode next sequential stream packet immediately into memory
            if not self.ret:  # Check if stream transmission failed or returned empty data structure packet assets
                self.stopped = True  # Toggle shutdown flag state to stop operations and initiate reconnection routines

    def read(self):  # Define public access function to return latest frame cache registry data instantly
        return self.ret, self.frame  # Return verification status boolean together with the most recent frame matrix layer

    def stop(self):  # Define stop function to handle clean teardown routines on hardware network interfaces
        self.stopped = True  # Toggle thread control flag state to terminate active background update loops instantly
        self.stream.release()  # Close network socket bindings and clear active system video capture handler channels

def load_dashboard_config():  # Declare a function to load updated line and polygon zone metrics directly from disk
    if os.path.exists(CONFIG_PATH):  # Check if the configuration JSON file is present on the local storage drive system
        try:  # Initialize a protective try-catch block to handle syntax errors or file access permission bugs
            with open(CONFIG_PATH, "r") as f:  # Open the configuration file in safe read-only text file mode
                return json.load(f)  # Parse the text file content into a native functional Python dictionary structure
        except Exception:  # Intercept any parsing errors to ensure pipeline operations do not crash unexpectedly
            pass  # Ignore the file error seamlessly and fall back to the default fallback configuration template below
    return {"counting_line_ratio": 0.50, "zones": []}  # Return default values if configuration file missing or corrupted

def write_telemetry(total_in, total_out, inside, fps):  # Declare a function to output current analytics metadata
    data = {  # Construct a dictionary containing all metrics updated in the current frame iteration loop
        "total_in": total_in,  # Map the accumulated inward crossing counter to the output payload data structure
        "total_out": total_out,  # Map the accumulated outward crossing counter to the output payload data structure
        "current_inside": inside,  # Map the count of unique objects within the designated polygon boundary lines
        "fps": round(fps, 1)  # Map the computed system processing speed rounded to one decimal place value
    }  # End of data payload dictionary instantiation pattern
    try:  # Open a safe execution block to prevent storage drive access failures from interrupting the system
        with open(TELEMETRY_PATH, "w") as f:  # Open the designated telemetry output file in write mode
            json.dump(data, f)  # Serialize and write the metrics dictionary as clean formatted json text data
    except Exception:  # Intercept file lock or disk capacity exceptions safely without freezing execution
        pass  # Bypass tracking file errors gracefully to prioritize live video stream rendering execution speed

print("[SUCCESS] Background core stream classes and telemetry helper utilities compiled successfully.")

[INFO] Flask Server Base Initialized ✅


# CELL 3 — MODEL INITIALIZATION

In [ ]:
print(f"[INFO] Loading YOLO model weights onto hardware layer: {DEVICE_NAME}...")

# Load the neural network architecture weights and force memory allocation onto the designated hardware accelerator device index
model = YOLO("yolo26l.pt", task="detect").to(f"cuda:{DEVICE_IDX}" if torch.cuda.is_available() else "cpu")

print("[SUCCESS] YOLO Model weights successfully initialized, allocated, and cached in system memory!")

[INFO] Loading MTCNN + FaceNet onto GPU...
[INFO] Face models loaded ✅ on NVIDIA GeForce RTX 4060



# CELL 4 — LINE & DIRECTION COUNTING LOGIC

In [ ]:
def process_counting_logic(frame, h, w, config, boxes, ids, track_history, crossed):
    """
    This function isolates all drawing and line-crossing mathematical logic parameters.
    Modify this function cell anytime to flip directions, line orientation, or tracking calculations.
    """
    global total_in, total_out  # Bring global counter variables into function scope to allow real-time value increments
    
    # ----------------------------------------------------------------------------------------------------
    # LINE ORIENTATION & POSITION SETUP (Default: Vertical Line splitting the screen width)
    # ----------------------------------------------------------------------------------------------------
    line_ratio = config.get("counting_line_ratio", 0.50)  # Extract the dynamic configuration multiplier value (defaults to 50%)
    line_pos_x = int(w * line_ratio)  # Calculate the absolute horizontal pixel coordinate position based on total frame width
    cv2.line(frame, (line_pos_x, 0), (line_pos_x, h), (0, 0, 255), 3)  # Render a solid red vertical barrier line across the screen height

    # ----------------------------------------------------------------------------------------------------
    # POLYGON ZONE BOUNDARY PARSING (Kept in background for config safety, but removed from visual header)
    # ----------------------------------------------------------------------------------------------------
    roi_polygon = None  # Declare an empty reference placeholder variable for the spatial region-of-interest object
    if config.get("zones"):  # Check if the external settings file provides valid coordinates for a regional layout polygon
        try:  # Wrap coordinates conversion inside a try-catch block to secure runtime thread execution from parsing crashes
            raw_pts = config["zones"][0]["points"]  # Extract the raw relative decimal tracking points matrix array from the config
            scaled_pts = np.array([[int(pt[0] * w), int(pt[1] * h)] for pt in raw_pts], dtype=np.int32)  # Scale point fractions to real frame sizes
            roi_polygon = scaled_pts.reshape((-1, 1, 2))  # Reshape the numpy array formatting parameters to fully support OpenCV compliance
            cv2.polylines(frame, [roi_polygon], True, (255, 0, 255), 2)  # Draw a persistent magenta outline polygon around the zone coordinates
        except Exception:  # Intercept formatting errors seamlessly without triggering system-wide execution halts
            roi_polygon = None  # Force reset the region-of-interest reference to None to bypass runtime orientation verification checks

    # ----------------------------------------------------------------------------------------------------
    # INDIVIDUAL OBJECT TRACKING AND COUNTING MATHEMATICS LOOP
    # ----------------------------------------------------------------------------------------------------
    for box, tid in zip(boxes, ids):  # Concurrently extract geometric boundary values and assigned unique instance index values
        cx = int((box[0] + box[2]) / 2)  # Compute the target entity's absolute geometric center pixel point on the horizontal x-axis
        cy = int((box[1] + box[3]) / 2)  # Compute the target entity's absolute geometric center pixel point on the vertical y-axis

        # ----------------------------------------------------------------------------------------------------
        # DIRECTION COUNTING LOGIC (Default: Left-to-Right = IN | Right-to-Left = OUT)
        # ----------------------------------------------------------------------------------------------------
        prev_cx = track_history.get(tid, cx)  # Query tracking coordinate cache records to fetch previous horizontal x-axis point location
        
        # Evaluate line-crossing spatial transition events triggers
        if prev_cx <= line_pos_x and cx > line_pos_x and crossed.get(tid) != "IN":  # Entity crossed red boundary barrier going rightwards
            total_in += 1  # Permanently increment the global inward entry accounting statistical counter variable
            crossed[tid] = "IN"  # Register the unique object tracking index status into the transition log dictionary to block double counts
        elif prev_cx >= line_pos_x and cx < line_pos_x and crossed.get(tid) != "OUT":  # Entity crossed red boundary barrier going leftwards
            total_out += 1  # Permanently increment the global outward exit accounting statistical counter variable
            crossed[tid] = "OUT"  # Register the unique object tracking index status into the transition log dictionary to block double counts

        # ----------------------------------------------------------------------------------------------------
        # VISUAL BOUNDING BOX DRAWING OVERLAYS
        # ----------------------------------------------------------------------------------------------------
        track_history[tid] = cx  # Cache current horizontal coordinate point into historical data registers for subsequent line crossing tests
        cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 0, 255), 2)  # Paint a blue bounding box around target coordinates
        cv2.putText(frame, f"ID #{tid}", (int(box[0]), int(box[1]) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)  # Print unique tracking assignment label

print("[SUCCESS] Counting and drawing logic function updated with new DETECTED and pure INSIDE mathematical configurations!")

[READY] Hybrid Engine Active ✅


# CELL 5 — MAIN PROCESSING ENGINE LOOP & LIVE DISPLAY

In [ ]:
# Reset runtime counting states on loop launch execution inside notebook workspace
total_in = 0  # Re-initialize inward counters back to absolute baseline zero value
total_out = 0  # Re-initialize outward counters back to absolute baseline zero value
track_history = {}  # Wipe coordinate trajectory caches completely to start tracking records anew
crossed = {}  # Clear crossing entry registries logs completely to handle fresh instance indexes
frame_count = 0  # Reset sequential performance monitoring counter markers integers back to zero

window_name = "AETHER VISION v5.0 - MODULAR IPYNB CORE"  # Define standard identifier string label for user UI window shell
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)  # Initialize resizable graphics visualization system container window

# Flag variable to explicitly control the multi-layered loop termination when 'q' is pressed
user_requested_exit = False  

while True:  # Enter infinite loop block to safely handle stream failures and connection retries
    if user_requested_exit:  # Check if the exit flag was set to True in the inner loop layer
        break  # Break the outer loop immediately to stop the entire cell execution safely
        
    print(f"[INFO] Connecting to pipeline feed thread instance: {UDP_URL}")  # Log connection state attempt markers
    vstream = RTSPVideoStream(UDP_URL).start()  # Spawn the parallel background stream tracking reader component framework
    time.sleep(1.0)  # Pause execution thread for exactly one second to allow network video socket frame data caches to initialize comfortably

    if not vstream.ret:  # Evaluate if background data consumer subsystem failed to acquire valid connection sockets
        print("[ERROR] Core connection failed to parse packets. Retrying in 3 seconds...")  # Output network alerts
        vstream.stop()  # Terminate and shut down the failed network frame worker sub-thread instance process safely
        time.sleep(3)  # Delay program execution for 3 full seconds before performing secondary connection routine tries
        continue  # Force execution loop pointer straight back to initialization line layers

    print("[SUCCESS] Connected! Streaming frame buffer live. Press 'q' inside the video box window to stop completely.")  # Confirm success logs
    config = load_dashboard_config()  # Initialize configuration variables mappings directly from the storage disk file system
    last_frame_time = time.time()  # Store current system timestamp as fresh benchmark reference entry point for heartbeat validation

    while True:  # Enter deep execution nested frame reading processing system iteration loop
        t0 = time.time()  # Capture precise start execution loop time metrics for performance profiling checks
        frame_count += 1  # Increment frame sequencing marker value variable by single digit integer values

        if frame_count % 10 == 0:  # Perform dynamic configuration synchronization check routines every 10 sequential cycles
            config = load_dashboard_config()  # Extract updated user configuration values safely without restarting stream channels

        ret, frame = vstream.read()  # Safely extract newest decoded frame array layer from background worker thread memory address locations
        if not ret or frame is None:  # Check if acquired frame matrix evaluates to broken state or completely empty registers
            if time.time() - last_frame_time > 3.0:  # Evaluate if total transmission drop duration metrics surpass 3 seconds bounds
                print("[WARNING] Connection timeout reached on streaming socket channels. Retrying network link...")  # Output alerts
                vstream.stop()  # Kill current background stream capture thread routines safely before falling back to outer recovery loop
                break  # Break frame tracking loops immediately to jump out into network socket retry phase blocks
            time.sleep(0.001)  # Sleep execution pipeline slightly to optimize machine cpu processing thread consumption rates
            continue  # Re-evaluate upcoming frame packets allocations cleanly

        last_frame_time = time.time()  # Update heartbeat watchdog verification clock value to signify incoming network feed is active
        h, w, _ = frame.shape  # Parse frame matrix dimensions configuration metadata values to read pixel height and width dimensions
        
        # Run standard neural network inference and bounding object tracking tracking layers using model memory objects loaded from Cell 3
        results = model.track(frame, imgsz=640, conf=0.25, classes=[0], tracker="bytetrack.yaml", persist=True, verbose=False, device=DEVICE_IDX)

        current_detected_count = 0  # Reset temporary frame-level visual headcount metrics values to zero
        
        if results[0].boxes.id is not None:  # Confirm tracking object framework successfully assigned valid index structures to objects in frame
            boxes = results[0].boxes.xyxy.cpu().numpy()  # Convert spatial bounding boxes coordinates tensors seamlessly into standard numpy arrays
            ids = results[0].boxes.id.cpu().numpy().astype(int)  # Convert unique object identifier tensors into integer arrays tracking vectors
            current_detected_count = len(ids)  # Capture the absolute number of active live boundary boxes visible in this precise frame
            
            # Call the independent spatial arithmetic logic processing function compiled inside Cell 4 registers
            _ = process_counting_logic(frame, h, w, config, boxes, ids, track_history, crossed)

        if len(track_history) > 300:  # Verify dictionary sizing boundaries properties criteria to trigger memory maintenance actions
            excess_keys = list(track_history.keys())[:-100]  # Isolate all historical identity keys index arrays records excluding the newest 100
            for k in excess_keys:  # Iterate over targets index array registers mappings one by one sequentially
                track_history.pop(k, None)  # Purge old coordinates locations index mapping metrics traces from ram memory structures safely
                crossed.pop(k, None)  # Purge matching orientation state triggers indicators values from dictionary arrays safely

        # CRITICAL MATHEMATICAL AMENDMENT: Compute the net remaining inside count by subtracting overall outs from total ins
        net_inside_count = max(0, total_in - total_out)  # Ensure math never outputs a negative value if edge anomalies happen
        
        fps = 1.0 / max(time.time() - t0, 0.001)  # Calculate frame execution calculation tracking speed cycle rating index values
        
        # Note: Sending the calculated net_inside_count seamlessly down to telemetry outputs
        write_telemetry(total_in, total_out, net_inside_count, fps)  

        # Draw translucent professional monitoring dashboard header layout overlays across visual screens panels top sections
        overlay = frame.copy()  # Duplicate active display canvas frame matrix layer data blocks to process modifications drawings separately
        cv2.rectangle(overlay, (0, 0), (w, 55), (20, 20, 20), -1)  # Render a dark status block header rectangle layer across total canvas width width
        cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)  # Alpha blend canvases matrices arrays layers to construct polished alpha transparency overlays

        # Format visual layout monitoring panel telemetry statistical metrics information data visualization text string lines strings
        header_text = f"IN: {total_in}  |  OUT: {total_out}  |  INSIDE: {net_inside_count}  |  DETECTED: {current_detected_count}  |  {round(fps, 1)} FPS"
        cv2.putText(frame, header_text, (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)  # Burn text values line onto display windows

        cv2.imshow(window_name, frame)  # Map fully updated frame image matrix layout values configurations directly onto specific user interface windows elements
        
        # Listen every 1ms for individual pressing of the 'q' keyboard character key inside the OpenCV window frame layout
        if cv2.waitKey(1) & 0xFF == ord('q'):  
            print("[INFO] 'q' key pressed. Initiating graceful multi-threaded shutdown pipeline...")  # Log shutdown to notebook terminal
            vstream.stop()  # Stop and release the parallel background worker thread to prevent memory leaks and zombie processes
            cv2.destroyAllWindows()  # Dismantle and close all graphical window screens managed by the OpenCV UI subsystem
            user_requested_exit = True  # Flip the boolean exit flag to True so the outer reconnect loop breaks immediately
            break  # Break out of the inner frame processing nested loop layer instantly

[DEPLOYMENT] Socket Listener Live on Port 8089 ✅


Exception in thread Thread-13 (socket_receiver_thread):
Traceback (most recent call last):
  File "c:\Users\Admin\anaconda3\envs\store_env\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\Admin\anaconda3\envs\store_env\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_908\3524174554.py", line 7, in socket_receiver_thread


OSError: [WinError 10048] Only one usage of each socket address (protocol/network address/port) is normally permitted
